In [1]:
from sr_model import single_sr, paired_sr
import os
from datasets import load_from_disk, concatenate_datasets
from datasets import Dataset
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader 


In [2]:

import yaml  # 导入 PyYAML 库

# 指定 config.yaml 文件的路径
config_path = os.path.join('/home/rsun@ZHANGroup.local/sr_project/configs/paired_configs/config.yaml')

# 读取 config.yaml 文件
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

# 打印配置内容以验证
print(config)

{'rna_config_path': '/home/rsun@ZHANGroup.local/sr_project/configs/rna_configs/config_3.yaml', 'ga_config_path': '/home/rsun@ZHANGroup.local/sr_project/configs/ga_configs/config_3.yaml', 'rna_checkpoint': '/home/rsun@ZHANGroup.local/sr_project/saved_models/config_3_2025-02-19-05-16/checkpoint_16000.pth', 'ga_checkpoint': '/home/rsun@ZHANGroup.local/sr_project/saved_models/ga_config3_2025-02-20-07-15/checkpoint_12000.pth', 'temperature': 0.1, 'tau': 10, 'freeze_param': True, 'learning_rate': 1e-05, 'weight_decay': 0.05, 'warmup_steps': 40, 'anneal_steps': 400, 'min_lr': 1e-07, 'log_dir': 'logs', 'save_dir': 'saved_models', 'run_name': 'paired_test', 'num_steps': 600, 'device': 'cuda', 'eval_steps': 40, 'save_steps': 100}


In [3]:
paired_model = paired_sr(config)

Initialize rna model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:139: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/config_3_2025-02-19-05-16/checkpoint_16000.pth at step 16000
Load rna checkpoint
Initialize ga model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:139: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/ga_config3_2025-02-20-07-15/checkpoint_12000.pth at step 12000
Load ga checkpoint
Initialize multi model
Freeze multi model param


In [4]:
paired_model.model 

multi_model(
  (rna_model): sr_single_omic(
    (encoder_net): feature_encoder(
      (encoder_header): Linear(in_features=32285, out_features=512, bias=True)
      (layernorm): LayerNorm((512,), eps=1e-08, elementwise_affine=True)
      (fc_layers): Sequential(
        (0): fc_net(
          (fc): Linear(in_features=512, out_features=256, bias=True)
          (act): LeakyReLU(negative_slope=0.01)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): fc_net(
          (fc): Linear(in_features=256, out_features=128, bias=True)
          (act): LeakyReLU(negative_slope=0.01)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (decoder_net): feature_decoder(
      (decoder_header): Linear(in_features=512, out_features=32285, bias=True)
      (fc_layers): Sequential(
        (0): fc_net(
          (fc): Linear(in_features=128, out_features=256, bias=True)
          (act): LeakyReLU(negative_slope=0.01)
          (dropout): Dropout(p=0.1, inpla

In [5]:
paired_model.set_optimizer()

(AdamW (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     initial_lr: 1e-05
     lr: 0.0
     maximize: False
     weight_decay: 0.05
 ),
 <custom_scheduler.WarmupCosineScheduler at 0x7cad70984370>)

: 